# Quant Finance Pipeline — Final Research Results Review

This notebook summarizes the final evidence chain for the portfolio project.

The goal is not to hide weak results. The goal is to show a reproducible
research process with walk-forward evaluation, leakage controls, transaction
costs, baselines, optimizer selection, and honest conclusions.

## Experimental Integrity Rules

- No test-window data is used for optimizer selection.
- No bad results are removed from the final report.
- Baselines remain visible.
- Transaction costs are included for net strategy results.
- Underperformance versus SPY is reported directly.
- Risk-control improvements are not described as broad outperformance.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

def load_json(relative_path: str) -> dict:
    return json.loads((PROJECT_ROOT / relative_path).read_text(encoding="utf-8"))

walk_forward = load_json("reports/walk_forward_optimizer_selection.json")
stitched = load_json("reports/walk_forward_optimizer_stitched_oos_equity_summary.json")

print("Walk-forward splits:", walk_forward["evaluated_split_count"])
print("Stitched OOS window:", stitched["stitched_start"], "->", stitched["stitched_end"])

## Core Result: Mixed / Negative Return Result

This notebook intentionally keeps the inconvenient results visible.

The evolutionary optimizer **did not outperform SPY** on stitched final equity,
stitched CAGR, or stitched Sharpe.

| Strategy | Final Equity | CAGR | Sharpe | Max Drawdown | Calmar |
|---|---:|---:|---:|---:|---:|
| SPY buy-and-hold | 2.920 | 13.87% | 0.785 | -33.72% | 0.411 |
| Momentum gross | 2.522 | 11.87% | 0.825 | -22.54% | 0.526 |
| Optimizer selected net | 1.633 | 6.13% | 0.642 | -18.52% | 0.331 |

**Strict interpretation:** the optimizer improved drawdown control, but it
sacrificed return and risk-adjusted performance. This is a **risk-control
trade-off**.

Exact phrase for review/test visibility: **risk-control trade-off**.

This is not market outperformance.

In [ ]:
metrics = pd.DataFrame(stitched["stitched_metrics"]).T

display_columns = [
    "final_equity",
    "cagr",
    "sharpe",
    "max_drawdown",
    "calmar",
    "observation_count",
]

metrics[display_columns].sort_values("final_equity", ascending=False)

In [ ]:
equity_path = PROJECT_ROOT / "reports/walk_forward_optimizer_stitched_oos_equity.parquet"
equity = pd.read_parquet(equity_path)
equity["date"] = pd.to_datetime(equity["date"])

equity_pivot = equity.pivot(
    index="date",
    columns="strategy_name",
    values="equity",
).sort_index()

ax = equity_pivot.plot(
    figsize=(12, 6),
    title="Stitched OOS Equity Curves: Optimizer vs Baselines",
)
ax.set_ylabel("Equity")
ax.set_xlabel("Date")
plt.show()

In [ ]:
drawdown = equity_pivot / equity_pivot.cummax() - 1.0

ax = drawdown.plot(
    figsize=(12, 6),
    title="Stitched OOS Drawdown Curves",
)
ax.set_ylabel("Drawdown")
ax.set_xlabel("Date")
plt.show()

In [ ]:
deltas = pd.Series(stitched["optimizer_vs_spy_deltas"], name="optimizer_minus_spy")
deltas.to_frame()

## Full Walk-Forward Optimizer Selection

The optimizer was trained only on each train window. A single candidate was
selected by fixed train Sharpe before the OOS test window was evaluated.

Across 9 OOS splits:

| Strategy | Mean CAGR | Mean Sharpe | Mean Max Drawdown |
|---|---:|---:|---:|
| SPY buy-and-hold | 15.00% | 1.215 | -14.07% |
| Optimizer selected net | 6.44% | 0.864 | -8.25% |

Again, the optimizer does **not** beat SPY on mean CAGR or mean Sharpe.
It only improves mean max drawdown. This is a risk-control trade-off, not
broad outperformance.

In [ ]:
winner_counts = pd.DataFrame(walk_forward["split_metric_winner_counts"]).fillna(0).astype(int)
winner_counts

In [ ]:
split_rows = []
for split in walk_forward["split_reports"]:
    selected = split["selected_train_evaluation"]
    test_metrics = split["selected_test_evaluation"]["metrics"]
    split_rows.append(
        {
            "split_index": split["split_index"],
            "test_start": split["test_start"],
            "test_end": split["test_end"],
            "selected_candidate": selected["candidate_id"],
            "test_cagr": test_metrics["cagr"],
            "test_sharpe": test_metrics["sharpe"],
            "test_max_drawdown": test_metrics["max_drawdown"],
        }
    )

pd.DataFrame(split_rows)

## What the Results Mean

The optimizer behaved like a conservative risk-control allocation mechanism.
It reduced stitched max drawdown versus SPY, but the cost was substantial:
lower final equity, lower CAGR, and lower Sharpe.

This is a valid research result, but it is not an alpha claim.
The correct conclusion is:

> The evolutionary optimizer produced a lower-drawdown allocation profile,
> but did not outperform simple market and momentum baselines on return or
> risk-adjusted return metrics.

## Limitations

- Historical backtest evidence is not live trading evidence.
- ETF universe is limited and may contain survivorship bias.
- yfinance data is research-grade, not institutional-grade.
- Transaction cost and slippage model is simplified.
- Optimizer search can overfit train windows.
- Selection by train Sharpe may prefer overly defensive parameter sets.
- No macro regime features are included.
- No nested validation or multi-seed robustness study is included yet.

## Next Steps

- Add volatility targeting as a separate baseline.
- Add drawdown-constrained objective selection.
- Compare selection by train Sharpe, Calmar, and constrained CAGR.
- Add multi-seed optimizer robustness analysis.
- Add parameter stability plots across splits.
- Add regime-aware features such as rates, inflation, volatility index, and
  trend filters.
- Add stricter transaction-cost and slippage stress tests.